
# Google Colab GGUF Conversion Notebook

This version is modified from the Kaggle notebook to run on Google Colab.

## What Changed
- Uses Google Drive instead of Kaggle Input
- Uses `/content` paths
- Avoids Kaggle 20GB storage limitation
- Saves GGUF directly to Google Drive
- Better for Unsloth GGUF conversion

## Before Running
1. Upload your adapter folder to Google Drive
2. Keep the adapter files together:
   - adapter_config.json
   - adapter_model.safetensors
   - tokenizer files
3. Update `DRIVE_ADAPTER_PATH` if needed


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# SENTINEL — Merge LoRA adapter into base + convert to GGUF

Takes the **already-trained SFT adapter** you have locally at
`C:\Users\niran\Desktop\Web_pentesting\sft_model\sft_adapter (1)\` and produces a
single quantized `.gguf` file ready to load into Ollama on your local 3050 4GB laptop.

## Run this on Kaggle, NOT locally
Your laptop has 4 GB VRAM + 15.8 GB RAM. The merge step needs ~16 GB RAM to hold the
base in BF16. Kaggle T4 (16 GB VRAM, 30 GB RAM) handles it cleanly.

## What you do before running
1. **Rename** the local folder `sft_adapter (1)` → `sft_adapter` (kill the parens + space —
   they break Kaggle dataset paths).
2. **Upload** that `sft_adapter/` folder to Kaggle as a new private dataset. Name it
   `sentinel-sft-adapter` (or whatever you like — update Cell 3 to match).
3. **Create a new Kaggle notebook**, Accelerator = `GPU T4 x1`, Internet = `On`,
   add the dataset via `+ Add Input`.
4. **Upload this `.ipynb`** to the notebook and run cells in order.

## Expected output
One file in `/kaggle/working/sentinel_gguf/`: `sentinel-llama3-8b-sft.Q4_K_M.gguf` (~4.6 GB).
Download from Kaggle's right-sidebar Output panel.

## Cell 1 · Install dependencies

`unsloth` provides the one-liner `save_pretrained_gguf`. `peft`/`bitsandbytes` are needed for adapter loading.

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q --upgrade "unsloth" peft accelerate bitsandbytes
!pip install -q huggingface_hub
print("deps installed")

## Cell 2 · Imports + GPU/disk check

Kaggle free tier gives ~20 GB on `/kaggle/working` and a separate ~20 GB on `/tmp`. Cell B (manual fallback) routes the 8 GB q8_0 intermediate + 1 GB llama.cpp build through `/tmp`, leaving `/kaggle/working` for the 16 GB merged-HF (briefly) and the ~5.4 GB final GGUF.

Need: **>=15 GB free on `/kaggle/working` and >=10 GB free on `/tmp`**.

In [ ]:
import os, gc, json, time, shutil, subprocess
from pathlib import Path
import torch
from huggingface_hub import login as hf_login

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Settings -> Accelerator -> GPU T4 x1 (or P100)")
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB)")

free_working_gb = shutil.disk_usage('/content/gguf_work').free / 1e9
free_tmp_gb     = shutil.disk_usage('/tmp').free / 1e9
print(f"Disk free on /content/gguf_work: {free_working_gb:.1f} GB")
print(f"Disk free on /tmp           : {free_tmp_gb:.1f} GB")

# Cell B routes the 8 GB q8_0 intermediate + 1 GB llama.cpp build through /tmp,
# so /content/gguf_work only briefly holds the 16 GB merged-HF, then the 4.6 GB final GGUF.
if free_working_gb < 15:
    raise RuntimeError(
        f"need >=15 GB free on /content/gguf_work for the merged-HF intermediate, "
        f"only {free_working_gb:.1f} GB available. Clean up Output panel or restart kernel."
    )
if free_tmp_gb < 10:
    raise RuntimeError(
        f"need >=10 GB free on /tmp for q8_0 intermediate + llama.cpp build, "
        f"only {free_tmp_gb:.1f} GB available."
    )
print("disk check passed")

PyTorch: 2.10.0+cu128
GPU: Tesla T4 (15.6 GB)


FileNotFoundError: [Errno 2] No such file or directory: '/content/gguf_work'

## Cell 3 · Configuration — **EDIT THE DATASET ROOT IF YOUR SLUG DIFFERS**

`DATASET_ROOT` points at the Kaggle-mounted dataset folder. The cell auto-discovers the adapter sub-folder by locating `adapter_config.json` inside it, so internal layout doesn't matter.

Quant choice for your 4 GB Laptop GPU:
- **`q5_k_m` (~5.4 GB) — recommended**: best quality on a 4 GB GPU, ~2-3 tok/s with partial CPU offload. Best for an SFT'd agent that must emit precise JSON.
- `q4_k_m` (~4.6 GB) — slightly faster, marginally lower JSON adherence.
- `q3_k_m` (~3.5 GB) — fits fully on GPU but degrades quality noticeably; skip for fine-tuned agent use.

In [ ]:

from pathlib import Path
import json

# ===== GOOGLE COLAB CONFIG =====
DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/sft_adapter"

GGUF_QUANT      = "q5_k_m"
PUSH_TO_HF      = False
HF_REPO_GGUF    = "YOUR-HF-USERNAME/sentinel-llama3-8b-sft-gguf"
HF_PRIVATE      = True
# ===============================

OUTPUT_DIR      = "/content/drive/MyDrive/gguf_output"
GGUF_OUT_DIR    = f"{OUTPUT_DIR}/sentinel_gguf"
MAX_SEQ_LENGTH  = 4096

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

dataset_root = Path(DRIVE_ADAPTER_PATH)

if not dataset_root.exists():
    raise FileNotFoundError(
        f"Adapter folder not found: {dataset_root}\n"
        f"Make sure your Google Drive path is correct."
    )

candidates = list(dataset_root.rglob("adapter_config.json"))

if not candidates:
    raise FileNotFoundError(
        f"No adapter_config.json found under {dataset_root}"
    )

adapter_dir = candidates[0].parent
ADAPTER_PATH = str(adapter_dir)

for required in ["adapter_config.json", "adapter_model.safetensors"]:
    p = adapter_dir / required
    assert p.exists(), f"missing required file: {p}"

print(f"adapter at: {adapter_dir}")

for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name:40s}  {f.stat().st_size/1e6:8.2f} MB")

with (adapter_dir / "adapter_config.json").open() as f:
    adapter_cfg = json.load(f)

BASE_MODEL = adapter_cfg["base_model_name_or_path"]

print(f"\nbase_model declared in adapter_config: {BASE_MODEL}")
print(f"  lora_rank      = {adapter_cfg.get('r')}")
print(f"  lora_alpha     = {adapter_cfg.get('lora_alpha')}")
print(f"  target_modules = {adapter_cfg.get('target_modules')}")


adapter at: /content/drive/MyDrive/sft_adapter
  README.md                                     0.01 MB
  adapter_config.json                           0.00 MB
  adapter_model.safetensors                   167.83 MB
  chat_template.jinja                           0.00 MB
  special_tokens_map.json                       0.00 MB
  tokenizer.json                               17.21 MB
  tokenizer_config.json                         0.05 MB
  training_args.bin                             0.01 MB

base_model declared in adapter_config: unsloth/llama-3-8b-Instruct-bnb-4bit
  lora_rank      = 16
  lora_alpha     = 32
  target_modules = ['v_proj', 'up_proj', 'k_proj', 'down_proj', 'q_proj', 'o_proj', 'gate_proj']


## Cell 4 · HuggingFace auth (only if pushing GGUF to HF)

Skip this cell if `PUSH_TO_HF = False`. Otherwise add `HF_TOKEN` to Kaggle Secrets first (sidebar → Add-ons → Secrets).

In [ ]:
hf_token = None
if PUSH_TO_HF:
    from kaggle_secrets import UserSecretsClient
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        raise RuntimeError(
            "HF_TOKEN secret missing. Add-ons -> Secrets -> Add a new secret\n"
            "  Label: HF_TOKEN   Value: hf_xxx... (must be a WRITE token)\n"
            f"Underlying error: {e}"
        )
    hf_login(token=hf_token, add_to_git_credential=True)
    print("HF authenticated")
else:
    print("HF push disabled (PUSH_TO_HF=False)")

HF push disabled (PUSH_TO_HF=False)


## Cell 5 · Load base + adapter from local path

`FastLanguageModel.from_pretrained(adapter_dir)` reads `adapter_config.json`, fetches the
base from `base_model_name_or_path` (downloaded from HF if not cached), and attaches the
LoRA. Loaded in 4-bit to fit T4 16 GB; the save step below upcasts internally.

In [ ]:
from unsloth import FastLanguageModel

t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Llama-3 has no default pad_token; set explicitly
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

FastLanguageModel.for_inference(model)
print(f"loaded in {time.time()-t0:.1f}s")
print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

Unsloth: Will load /content/drive/MyDrive/sft_adapter as a legacy tokenizer.
Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


loaded in 105.2s
VRAM after load: 5.89 GB / 15.64 GB


## Cell 6 · Smoke test the adapter before spending 20+ minutes converting

If this generation is NOT a valid SENTINEL JSON (Thought/Action/Action_Input), STOP — the
adapter didn't load correctly and converting it would be wasted work.

In [ ]:
SENTINEL_SYSTEM = (
    "You are SENTINEL, an autonomous web-exploitation agent. Given an HTML snippet and a goal "
    "(and optionally prior agent turns), you reason about vulnerabilities and emit a single "
    "JSON action that advances the exploit loop:\n\n"
    "observe -> identify attack surface -> select exploit -> generate payload -> interpret "
    "response -> adapt and retry -> detect success -> STOP\n\n"
    "Prioritize vulnerability sinks: form action, input value, src, href, hidden fields, query "
    "parameters, JSON body fields, and reflected DOM contexts. Infer the backend from HTML "
    "evidence (.php, .aspx, __VIEWSTATE, .jsp, /rest/, <app-*>, wp-content, etc.) and choose "
    "context-appropriate payloads.\n\n"
    "Output a single JSON object with exactly these keys:\n"
    "- Thought: <=4 sentences, <=80 words; cite the specific sink, backend inference, injection "
    "context, and payload-class justification (or signal classification for ANALYZE_RESPONSE / "
    "success indicator for STOP).\n"
    "- Action: one of SQL_INJECT | XSS_INJECT | RETRY_MUTATED | ANALYZE_RESPONSE | CRAWL_DEEPER "
    "| WAIT | STOP.\n"
    "- Action_Input: object with target_url, method, parameters, headers, rationale, plus "
    "action-specific fields (mutation_class for RETRY_MUTATED; signal + next_recommended for "
    "ANALYZE_RESPONSE; success_state + evidence for STOP).\n\n"
    "Output ONLY the JSON. No prose, no markdown fences, no commentary."
)

test_msgs = [
    {"role": "system", "content": SENTINEL_SYSTEM},
    {"role": "user", "content": (
        "GOAL: Bypass authentication and reach admin dashboard\n\n"
        "HTML_SNIPPET:\n"
        '<form method="POST" action="/login.php" id="loginForm">'
        '<input type="text" name="username" autocomplete="off">'
        '<input type="password" name="password">'
        '<input type="hidden" name="redirect" value="/admin/dashboard">'
        '<button type="submit">Sign In</button></form>'
    )},
]
prompt = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== ADAPTER SMOKE-TEST OUTPUT ===")
print(gen)
print("\n--- Sanity check ---")
try:
    parsed = json.loads(gen.strip().strip('`').replace('```json', '').replace('```', ''))
    ok = isinstance(parsed, dict) and {"Thought", "Action", "Action_Input"}.issubset(parsed.keys())
    print(f"JSON parses: {True}")
    print(f"Has Thought/Action/Action_Input keys: {ok}")
    print(f"Action value: {parsed.get('Action')!r}")
    if not ok:
        print("\nWARNING: output is JSON but missing expected keys. Adapter may be misaligned.")
except Exception as e:
    print(f"JSON parse failed: {e}")
    print("WARNING: do NOT proceed to conversion if this didn't produce SENTINEL JSON.")

Both `max_new_tokens` (=400) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

=== ADAPTER SMOKE-TEST OUTPUT ===
{"Thought":"POST /login.php with username text input — PHP/MySQL backend. SQL-string context. OR-tautology in username with comment terminator; redirect preserved.","Action":"SQL_INJECT","Action_Input":{"target_url":"/login.php","method":"POST","parameters":{"username":"admin' OR '1'='1' -- ","password":"x","redirect":"/admin/dashboard"},"headers":{"Content-Type":"application/x-www-form-urlencoded"},"rationale":"MySQL tautology bypasses password check; redirect preserved"}}

--- Sanity check ---
JSON parses: True
Has Thought/Action/Action_Input keys: True
Action value: 'SQL_INJECT'


## Cell 7 · CELL A — Merge + convert + quantize (the one-shot path)

> ⚠️ **On Kaggle free tier (20 GB on `/kaggle/working`), Cell A will almost certainly fail with `No space left on device`** because Unsloth's `save_pretrained_gguf` keeps both the 16 GB merged-HF and the 16 GB BF16 GGUF on the same filesystem at the same time. **Skip directly to Cell 8 (Cell B)** — it routes the intermediate through `/tmp` so it fits.
>
> Keep Cell A here only as a reference for what `save_pretrained_gguf` would do if you had ~50 GB of headroom (e.g. Kaggle Pro, Colab Pro, a local 24 GB SSD).

If you still want to try it: it does merge → convert → quantize in one call, ~20-30 min on T4 when disk allows. Otherwise → Cell 8.

In [ ]:
from pathlib import Path
import shutil
import time

# Where GGUF will finally live in Drive
DRIVE_SAVE_DIR = "/content/drive/MyDrive/sentinel_gguf"

Path(DRIVE_SAVE_DIR).mkdir(parents=True, exist_ok=True)

t0 = time.time()

# Save locally first (faster)
model.save_pretrained_gguf(
    GGUF_OUT_DIR,
    tokenizer,
    quantization_method=GGUF_QUANT,
)

print(f"\nGGUF write done in {(time.time()-t0)/60:.1f} min")

!ls -lh {GGUF_OUT_DIR}

# Find produced GGUF
gguf_files = list(Path(GGUF_OUT_DIR).glob("*.gguf"))

assert gguf_files, f"no .gguf file produced in {GGUF_OUT_DIR}"

biggest = max(gguf_files, key=lambda p: p.stat().st_size)

size_gb = biggest.stat().st_size / 1e9

print(f"\nFinal GGUF: {biggest} ({size_gb:.2f} GB)")

assert size_gb > 2.0, (
    f"GGUF suspiciously small ({size_gb:.2f} GB)"
)

# Copy to Google Drive
dest = Path(DRIVE_SAVE_DIR) / biggest.name

print(f"\nCopying GGUF to Google Drive...")
shutil.copy2(biggest, dest)

print(f"\nSaved to:")
print(dest)

print(f"\nDone.")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/gguf_output/sentinel_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [09:34<00:00, 143.63s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/gguf_output/sentinel_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile llama-3-8b-instruct.F16.gguf --outtype f16 --split-max-size 50G /content/drive/MyDrive/gguf_output/sentinel_gguf`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'llama-3-8b-instruct.F16.gguf', '--outtype', 'f16', '--split-max-size', '50G', '/content/drive/MyDrive/gguf_output/sentinel_gguf']' returned non-zero exit status 1.

In [ ]:
!ls /content/drive/MyDrive/gguf_output/sentinel_gguf

chat_template.jinja		  model-00004-of-00004.safetensors
config.json			  model.safetensors.index.json
model-00001-of-00004.safetensors  tokenizer_config.json
model-00002-of-00004.safetensors  tokenizer.json
model-00003-of-00004.safetensors


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!cmake -B build
!cmake --build build --config Release -j 8

Cloning into 'llama.cpp'...
remote: Enumerating objects: 93144, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 93144 (delta 87), reused 58 (delta 57), pack-reused 93002 (from 3)
Receiving objects: 100% (93144/93144), 386.55 MiB | 28.73 MiB/s, done.
Resolving deltas: 100% (66152/66152), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification i

In [ ]:
!python convert_hf_to_gguf.py \
    /content/drive/MyDrive/gguf_output/sentinel_gguf \
    --outfile /content/model-f16.gguf \
    --outtype f16


INFO:hf-to-gguf:Loading model: sentinel_gguf
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
Traceback (most recent call last):
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14230, in <module>
    main()
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14206, in main
    model_instance = model_class(dir_model, output_type, fname_out,
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 2864, in __init__
    super().__init__(*args, **kwargs)
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 1027, in __init__
    super().__init__(*args, **kwargs)
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 143, in __init__
    self.model_tensors = self.index_tensors(remote_hf_model_id=re

In [ ]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py \
    /content/drive/MyDrive/gguf_output/sentinel_gguf \
    --outfile /content/model-f16.gguf \
    --outtype f16

/content/llama.cpp
INFO:hf-to-gguf:Loading model: sentinel_gguf
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
Traceback (most recent call last):
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14230, in <module>
    main()
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14206, in main
    model_instance = model_class(dir_model, output_type, fname_out,
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 2864, in __init__
    super().__init__(*args, **kwargs)
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 1027, in __init__
    super().__init__(*args, **kwargs)
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 143, in __init__
    self.model_tensors = self.index_tensors(re

In [ ]:
!cp -r /content/drive/MyDrive/gguf_output/sentinel_gguf /content/

In [ ]:
!ls -lh /content/sentinel_gguf

total 11G
-rw------- 1 root root  536 May 12 12:20 chat_template.jinja
-rw------- 1 root root  809 May 12 12:20 config.json
-rw------- 1 root root    0 May 12 12:20 model-00001-of-00004.safetensors
-rw------- 1 root root 4.7G May 12 12:21 model-00002-of-00004.safetensors
-rw------- 1 root root 4.6G May 12 12:22 model-00003-of-00004.safetensors
-rw------- 1 root root 1.1G May 12 12:23 model-00004-of-00004.safetensors
-rw------- 1 root root  24K May 12 12:20 model.safetensors.index.json
-rw------- 1 root root  51K May 12 12:20 tokenizer_config.json
-rw------- 1 root root  17M May 12 12:20 tokenizer.json


In [ ]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py \
    /content/sentinel_gguf \
    --outfile /content/model-f16.gguf \
    --outtype f16

/content/llama.cpp
INFO:hf-to-gguf:Loading model: sentinel_gguf
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
Traceback (most recent call last):
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14230, in <module>
    main()
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14206, in main
    model_instance = model_class(dir_model, output_type, fname_out,
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 2864, in __init__
    super().__init__(*args, **kwargs)
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 1027, in __init__
    super().__init__(*args, **kwargs)
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 143, in __init__
    self.model_tensors = self.index_tensors(re

In [ ]:
from safetensors import safe_open
from pathlib import Path

model_dir = Path("/content/sentinel_gguf")

for f in sorted(model_dir.glob("*.safetensors")):
    print(f"\nChecking: {f.name}")

    try:
        with safe_open(f, framework="pt") as sf:
            keys = sf.keys()
            print("OK:", len(list(keys)), "tensors")
    except Exception as e:
        print("BROKEN:", e)


Checking: model-00001-of-00004.safetensors
BROKEN: Error while deserializing header: header too small

Checking: model-00002-of-00004.safetensors
OK: 104 tensors

Checking: model-00003-of-00004.safetensors
OK: 100 tensors

Checking: model-00004-of-00004.safetensors
OK: 5 tensors


In [ ]:
from pathlib import Path

MERGED_DIR = "/content/merged_model"

Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

print("Saving merged model locally...")

model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="5GB"
)

tokenizer.save_pretrained(MERGED_DIR)

print("Done.")

Saving merged model locally...


Unsloth: Restored added_tokens_decoder metadata in /content/merged_model/tokenizer_config.json.


Done.


In [ ]:
from safetensors import safe_open
from pathlib import Path

for f in sorted(Path("/content/merged_model").glob("*.safetensors")):
    print(f"\nChecking: {f.name}")

    try:
        with safe_open(f, framework="pt") as sf:
            print("OK:", len(list(sf.keys())), "tensors")

    except Exception as e:
        print("BROKEN:", e)


Checking: adapter_model.safetensors
OK: 448 tensors


In [ ]:
!rm -rf /content/merged_model

In [ ]:
from pathlib import Path

MERGED_DIR = "/content/merged_model"

Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

print("Saving merged model locally...")

model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="5GB"
)

tokenizer.save_pretrained(MERGED_DIR)

print("Merge save complete.")

Saving merged model locally...
Merge save complete.


In [ ]:
from safetensors import safe_open
from pathlib import Path

for f in sorted(Path("/content/merged_model").glob("*.safetensors")):
    print(f"\nChecking: {f.name}")

    try:
        with safe_open(f, framework="pt") as sf:
            print("OK:", len(list(sf.keys())), "tensors")

    except Exception as e:
        print("BROKEN:", e)


Checking: adapter_model.safetensors
OK: 448 tensors


In [ ]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py \
    /content/merged_model \
    --outfile /content/model-f16.gguf \
    --outtype f16

/content/llama.cpp
INFO:hf-to-gguf:Loading model: merged_model
Traceback (most recent call last):
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 973, in load_hparams
    config = AutoConfig.from_pretrained(dir_model, trust_remote_code=False).to_dict()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/configuration_auto.py", line 1528, in from_pretrained
    raise ValueError(
ValueError: Unrecognized model in /content/merged_model. Should have a `model_type` key in its config.json.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14230, in <module>
    main()
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 14189, in main
    hparams = ModelBase.load_hparams(dir_model, is_mistral_format)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/c

In [ ]:
!ls -lh /content/merged_model

total 177M
-rw-r--r-- 1 root root 1.3K May 12 12:28 adapter_config.json
-rw-r--r-- 1 root root 161M May 12 12:28 adapter_model.safetensors
-rw-r--r-- 1 root root  551 May 12 12:28 chat_template.jinja
-rw-r--r-- 1 root root 5.2K May 12 12:28 README.md
-rw-r--r-- 1 root root  50K May 12 12:29 tokenizer_config.json
-rw-r--r-- 1 root root  17M May 12 12:28 tokenizer.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r \
/content/drive/MyDrive/sft_adapter/ \
/content/adapter_clean

In [ ]:
from safetensors import safe_open
from pathlib import Path

f = Path("/content/adapter_clean/adapter_model.safetensors")

with safe_open(f, framework="pt") as sf:
    print("OK:", len(list(sf.keys())), "tensors")

OK: 448 tensors


In [ ]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ab7lznvf/unsloth_300a9b66da7e408c9a41fcd58f82e1a4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ab7lznvf/unsloth_300a9b66da7e408c9a41fcd58f82e1a4
  Resolved https://github.com/unslothai/unsloth.git to commit 040b80a60e6bf0278c0b81ee986cf9b87b9d4f6d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 31.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 61.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 85.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/adapter_clean",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

Unsloth: Will load /content/adapter_clean as a legacy tokenizer.
Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
model.save_pretrained_merged(
    "/content/merged_model",
    tokenizer,
    save_method="merged_16bit"
)

Unsloth: Restored added_tokens_decoder metadata in /content/merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [05:57<00:00, 89.47s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:05<00:00, 61.36s/it]


Unsloth: Merge process complete. Saved to `/content/merged_model`


In [ ]:
!ls -lh /content/merged_model

total 15G
-rw-r--r-- 1 root root  551 May 12 13:06 chat_template.jinja
-rw-r--r-- 1 root root  809 May 12 13:06 config.json
-rw-r--r-- 1 root root 4.7G May 12 13:13 model-00001-of-00004.safetensors
-rw-r--r-- 1 root root 4.7G May 12 13:14 model-00002-of-00004.safetensors
-rw-r--r-- 1 root root 4.6G May 12 13:16 model-00003-of-00004.safetensors
-rw-r--r-- 1 root root 1.1G May 12 13:16 model-00004-of-00004.safetensors
-rw-r--r-- 1 root root  24K May 12 13:06 model.safetensors.index.json
-rw-r--r-- 1 root root  51K May 12 13:06 tokenizer_config.json
-rw-r--r-- 1 root root  17M May 12 13:06 tokenizer.json


In [ ]:
from safetensors import safe_open
from pathlib import Path

for f in sorted(Path("/content/merged_model").glob("*.safetensors")):
    print(f"\nChecking: {f.name}")

    with safe_open(f, framework="pt") as sf:
        print("OK:", len(list(sf.keys())), "tensors")


Checking: model-00001-of-00004.safetensors
OK: 82 tensors

Checking: model-00002-of-00004.safetensors
OK: 104 tensors

Checking: model-00003-of-00004.safetensors
OK: 100 tensors

Checking: model-00004-of-00004.safetensors
OK: 5 tensors


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!cmake -B build
!cmake --build build --config Release -j 8

Cloning into 'llama.cpp'...
remote: Enumerating objects: 92822, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 92822 (delta 89), reused 58 (delta 57), pack-reused 92673 (from 3)
Receiving objects: 100% (92822/92822), 387.26 MiB | 32.09 MiB/s, done.
Resolving deltas: 100% (65919/65919), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification i

In [ ]:
!python convert_hf_to_gguf.py \
    /content/merged_model \
    --outfile /content/model-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {4096, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {4096, 14336

In [ ]:
!./build/bin/llama-quantize \
    /content/model-f16.gguf \
    /content/model-q5_k_m.gguf \
    q5_k_m

llama_print_build_info: build = 9121 (89730c8d2)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/model-f16.gguf' to '/content/model-q5_k_m.gguf' as Q5_K_M
llama_model_loader: loaded meta data with 29 key-value pairs and 291 tensors from /content/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 8.0B
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                       llama.context_length u32   

In [ ]:
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/sentinel_gguf"

Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
!cp /content/model-q5_k_m.gguf \
    /content/drive/MyDrive/sentinel_gguf/

In [ ]:
!ls -lh /content/drive/MyDrive/sentinel_gguf

total 5.4G
-rw------- 1 root root 5.4G May 12 14:05 model-q5_k_m.gguf


In [ ]:
!mkdir -p /content/drive/MyDrive/sentinel_gguf

!cp /content/model-q5_k_m.gguf \
    /content/drive/MyDrive/sentinel_gguf/

In [ ]:
!cp /content/model-q5_k_m.gguf \
    /content/drive/MyDrive/sentinel_gguf/

## Cell 8 · CELL B — Manual fallback (only if Cell 7 failed)

Same outcome, explicit steps. Easier to debug — you can see exactly which step failed.

Skip this cell if Cell 7 already produced a .gguf file.

In [ ]:
# Disk-aware path optimized for Kaggle free tier (~20 GB on /content/gguf_work).
# Routes the q8_0 intermediate (~8 GB) + llama.cpp build (~1 GB) to /tmp so
# /content/gguf_work only briefly holds the 16 GB merged-HF, then the 4.6 GB final.

MERGED_DIR  = f"{OUTPUT_DIR}/sentinel_merged"        # 16 GB, deleted after step 4
LCPP_DIR    = "/content/tmp/llama.cpp"                       # build lives in /tmp
TMP_GGUF    = "/content/tmp/sentinel-q8_0.gguf"              # 8 GB intermediate in /tmp
QUANT_GGUF  = f"{OUTPUT_DIR}/sentinel-{GGUF_QUANT}.gguf"  # 4.6 GB final
QUANT_UPPER = GGUF_QUANT.upper()

def _free_gb(path):
    return shutil.disk_usage(path).free / 1e9

print(f"start: working={_free_gb('/content/gguf_work'):.1f} GB free, "
      f"/tmp={_free_gb('/tmp'):.1f} GB free")

# ─── 1. Merge LoRA into base → 16-bit HF model (~16 GB on /content/gguf_work) ───
t0 = time.time()
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"[1] merged HF -> {MERGED_DIR}  ({(time.time()-t0)/60:.1f} min)")
print(f"    after merge: working={_free_gb('/content/gguf_work'):.1f} GB free")
!ls -lh {MERGED_DIR}

# ─── 2. Clone llama.cpp into /tmp + install conversion deps ───
if not os.path.exists(LCPP_DIR):
    !git clone --depth=1 https://github.com/ggerganov/llama.cpp {LCPP_DIR}
# requirements.txt is broadest; safer across llama.cpp versions
!pip install -q -r {LCPP_DIR}/requirements.txt
print("[2] llama.cpp cloned to /tmp + Python deps installed")

# ─── 3. Convert HF -> q8_0 GGUF directly into /tmp (~8 GB; ~half of bf16) ───
# q8_0 is a lossless-enough source for further quantization to Q4_K_M.
t0 = time.time()
!python {LCPP_DIR}/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {TMP_GGUF} \
    --outtype q8_0
print(f"[3] convert_hf_to_gguf -> q8_0 in {(time.time()-t0)/60:.1f} min")
!ls -lh {TMP_GGUF}

# ─── 4. Free 16 GB on /content/gguf_work by deleting merged-HF ───
# (q8_0 GGUF in /tmp has everything we still need)
!rm -rf {MERGED_DIR}
print(f"[4] merged-HF deleted; working={_free_gb('/content/gguf_work'):.1f} GB free")

# ─── 5. Build llama-quantize (CUDA preferred; fall back to CPU build) ───
t0 = time.time()
build_result = subprocess.run(
    f"cd {LCPP_DIR} && cmake -B build -DGGML_CUDA=ON -DLLAMA_CURL=OFF "
    f"&& cmake --build build --config Release -j --target llama-quantize",
    shell=True, capture_output=True, text=True,
)
if build_result.returncode != 0:
    print("CUDA build failed; falling back to CPU build")
    print(build_result.stderr[-500:])
    subprocess.run(
        f"cd {LCPP_DIR} && cmake -B build_cpu -DLLAMA_CURL=OFF "
        f"&& cmake --build build_cpu --config Release -j --target llama-quantize",
        shell=True, check=True,
    )

QUANT_BIN = (
    f"{LCPP_DIR}/build/bin/llama-quantize"
    if os.path.exists(f"{LCPP_DIR}/build/bin/llama-quantize")
    else f"{LCPP_DIR}/build_cpu/bin/llama-quantize"
)
assert os.path.exists(QUANT_BIN), f"llama-quantize binary not found at {QUANT_BIN}"
print(f"[5] llama-quantize built in {(time.time()-t0)/60:.1f} min  @ {QUANT_BIN}")

# ─── 6. Quantize q8_0 -> final K-quant, output to /content/gguf_work ───
t0 = time.time()
!{QUANT_BIN} {TMP_GGUF} {QUANT_GGUF} {QUANT_UPPER}
print(f"[6] quantize done in {(time.time()-t0)/60:.1f} min")

# ─── 7. Reclaim all /tmp space; keep only the final GGUF in /content/gguf_work ───
!rm -rf {TMP_GGUF} {LCPP_DIR}
!ls -lh {QUANT_GGUF}

size_gb = Path(QUANT_GGUF).stat().st_size / 1e9
print(f"\nFinal GGUF: {QUANT_GGUF}  ({size_gb:.2f} GB)")
assert size_gb > 2.0, f"GGUF suspiciously small ({size_gb:.2f} GB) — investigate before using"
print(f"end: working={_free_gb('/content/gguf_work'):.1f} GB free")

## Cell 9 · Download + Ollama handoff (read this last)

1. **Download the .gguf**:
   - Kaggle right sidebar → Output panel → expand the file → click the download arrow.
   - Or via HF if you set `PUSH_TO_HF=True`:
     `huggingface-cli download YOUR-HF-USERNAME/sentinel-llama3-8b-sft-gguf --local-dir ./gguf`

2. **On your Windows laptop**, install Ollama from https://ollama.com/download/windows

3. **Create a `Modelfile`** in the same folder as the .gguf:

   ```
   FROM ./sentinel-llama3-8b-sft.Q4_K_M.gguf

   TEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

   {{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

   {{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

   """
   PARAMETER stop "<|eot_id|>"
   PARAMETER stop "<|end_of_text|>"
   PARAMETER num_ctx 4096
   PARAMETER temperature 0.0
   ```

4. **Register + run**:
   ```powershell
   ollama create sentinel -f Modelfile
   ollama run sentinel
   ```

5. **Verify the OpenAI-compatible endpoint**:
   ```powershell
   curl http://localhost:11434/v1/chat/completions -H "Content-Type: application/json" -d '{"model":"sentinel","messages":[{"role":"user","content":"hi"}]}'
   ```

After that's confirmed working, come back to me — we still need to bridge the SENTINEL
JSON output format (Thought/Action/Action_Input) to the agent's tool layer
(NAVIGATE/INJECT_PAYLOAD/etc.) before the agent loop can use the model.

In [ ]:
!cp /content/model-f16.gguf \
    /content/drive/MyDrive/sentinel_gguf/

In [ ]:
!ls -lh /content/drive/MyDrive/sentinel_gguf